# Suite2p — Mesoscope (MROI) Pipeline

Run suite2p on a ScanImage **multi-ROI (mesoscope)** acquisition.

1. Load your **saved ops file** (`.json` or `ops.npy`) — your standard processing parameters.
2. Load the `ops.json` written by the MATLAB helper [`mesoscope_json_from_scanimage.m`](https://github.com/MouseLand/suite2p/blob/master/helpers/mesoscope_json_from_scanimage.m).
3. Overwrite **only** the recording-specific fields computed by that script: `fs`, `nplanes`, `nrois`, `mesoscan`, `lines`, `dx`, `dy`. Everything else in the MATLAB JSON (batch_size, nonrigid, tau, paths, …) is ignored, so your ops file stays in control.
4. Run `run_s2p(ops, db)`.

In [ ]:
import json
import numpy as np
from pathlib import Path
import suite2p
from suite2p import run_s2p, default_ops

print(f"suite2p version: {suite2p.__version__}")

## 1. Load your ops file

In [ ]:
OPS_FILE = r"/path/to/suite2p_ops_used.json"   # or ops.npy from any planeX/ folder

ops_path = Path(OPS_FILE)
assert ops_path.exists(), f"Ops file not found: {ops_path}"

if ops_path.suffix == ".json":
    with open(ops_path) as f:
        ops_loaded = json.load(f)
elif ops_path.suffix == ".npy":
    ops_loaded = np.load(ops_path, allow_pickle=True).item()
else:
    raise ValueError(f"Unsupported format: {ops_path.suffix}")

# Fill in any keys missing from the file (e.g. added in newer suite2p versions).
# Keys in your file take priority over defaults.
ops = default_ops()
ops.update(ops_loaded)

print(f"Loaded {len(ops_loaded)} parameters from: {ops_path.name}")

## 2. Load the MATLAB-generated MROI JSON

Only these seven fields are taken from it:

| key | meaning |
|---|---|
| `fs` | frame rate (Hz) |
| `nplanes` | number of z-planes |
| `nrois` | number of ScanImage ROIs |
| `mesoscan` | 1 = multi-ROI recording |
| `lines` | per ROI, the row indices of the raw (interleaved) frame belonging to that ROI |
| `dx`, `dy` | per ROI, x / y pixel offset of the ROI in the stitched field of view |

In [ ]:
MESO_JSON = r"/path/to/session/ops.json"   # written by mesoscope_json_from_scanimage.m

assert Path(MESO_JSON).exists(), f"MROI JSON not found: {MESO_JSON}"
with open(MESO_JSON) as f:
    meso = json.load(f)

MESO_KEYS = ["fs", "nplanes", "nrois", "mesoscan", "lines", "dx", "dy"]
missing = [k for k in MESO_KEYS if k not in meso]
assert not missing, f"Missing keys in {MESO_JSON}: {missing}"

print("Fields taken from the MROI JSON:")
for k in MESO_KEYS:
    v = meso[k]
    print(f"  {k:9s} = {'%d ROI row-arrays, lengths %s' % (len(v), [len(r) for r in v]) if k == 'lines' else v}")

## 3. Overwrite the MROI-specific fields

In [ ]:
changes = {}
def setop(key, value):
    changes[key] = (ops.get(key, "<unset>"), value)
    ops[key] = value

setop("fs",       float(meso["fs"]))
setop("nplanes",  int(meso["nplanes"]))
setop("nrois",    int(meso["nrois"]))
setop("mesoscan", bool(meso["mesoscan"]))
setop("lines",    [list(map(int, r)) for r in meso["lines"]])
setop("dx",       [int(v) for v in np.atleast_1d(meso["dx"])])
setop("dy",       [int(v) for v in np.atleast_1d(meso["dy"])])

print("Overwritten:")
for k, (old, new) in changes.items():
    show = lambda x: f"<{len(x)} ROI row-arrays>" if k == "lines" and isinstance(x, list) else x
    print(f"  {k:9s} {str(show(old)):>16}  ->  {show(new)}")

## 4. Sanity checks

In [ ]:
nrois, lines = ops["nrois"], ops["lines"]
assert ops["mesoscan"], "mesoscan is False — this JSON does not describe a multi-ROI recording"
assert len(lines) == len(ops["dx"]) == len(ops["dy"]) == nrois, \
    f"nrois={nrois}, len(lines)={len(lines)}, len(dx)={len(ops['dx'])}, len(dy)={len(ops['dy'])}"

print(f"nrois={nrois}  nplanes={ops['nplanes']}  fs={ops['fs']} Hz\n")
print("ROI layout (rows in raw frame -> offset in stitched FoV):")
for i, rows in enumerate(lines):
    print(f"  ROI {i}: rows {rows[0]}-{rows[-1]} ({len(rows)} rows)   dy={ops['dy'][i]}  dx={ops['dx'][i]}")

## 5. Database

`data_path` and `save_path0` are set here, per session (they are not taken from the MATLAB JSON).

> **GPU memory:** MROI frames are large. If your ops file has a `batch_size` that was tuned for 512×512 data it may run out of VRAM on a 12 GB GPU with `nonrigid=True` — see the `batch_size` notes in `suite2p_pipeline.ipynb` (empirically ~30 for 1000×2000 frames).

In [ ]:
db = {
    "data_path":  [r"/path/to/session/tiffs"],
    "save_path0": r"/path/to/session/results",
    # "fast_disk": r"/scratch",
}
for p in db["data_path"]:
    assert Path(p).exists(), f"data_path not found: {p}"
Path(db["save_path0"]).mkdir(parents=True, exist_ok=True)

print(f"batch_size = {ops['batch_size']}   nonrigid = {ops['nonrigid']}   block_size = {ops['block_size']}")

## 6. Run the pipeline

In [ ]:
output_ops = run_s2p(ops=ops, db=db)
print("suite2p finished")

## 7. Inspect results

For mesoscope data suite2p writes one `planeN` folder per ROI × plane, plus a `combined` folder with everything stitched into the field of view.

In [ ]:
import matplotlib.pyplot as plt

suite2p_dir = Path(db["save_path0"]) / "suite2p"
print("Folders:", sorted(d.name for d in suite2p_dir.iterdir() if d.is_dir()))

result_dir = suite2p_dir / "combined" if (suite2p_dir / "combined").exists() else sorted(suite2p_dir.glob("plane*"))[0]
print("Loading from:", result_dir.name)

ops_out = np.load(result_dir / "ops.npy",    allow_pickle=True).item()
F       = np.load(result_dir / "F.npy",      allow_pickle=True)
Fneu    = np.load(result_dir / "Fneu.npy",   allow_pickle=True)
iscell  = np.load(result_dir / "iscell.npy", allow_pickle=True)
cell_idx = np.where(iscell[:, 0].astype(bool))[0]

print(f"ROIs: {F.shape[0]}   cells: {len(cell_idx)}   frames: {F.shape[1]}   duration: {F.shape[1]/ops_out['fs']:.1f} s")

img = ops_out["meanImg"]
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(img, cmap="gray", vmin=np.percentile(img, 1), vmax=np.percentile(img, 99))
ax.set_title(f"Mean image ({result_dir.name})"); ax.axis("off")
plt.show()

In [ ]:
n = min(20, len(cell_idx))
sel = np.random.default_rng(0).choice(cell_idx, size=n, replace=False)
Fc  = F[sel] - ops["neucoeff"] * Fneu[sel]
F0  = np.percentile(Fc, 8, axis=1, keepdims=True)
dFF = (Fc - F0) / (F0 + 1)
t   = np.arange(F.shape[1]) / ops_out["fs"]

fig, ax = plt.subplots(figsize=(16, n * 0.5 + 1))
for i, tr in enumerate(dFF):
    ax.plot(t, tr + i * 3, lw=0.6, color="k")
ax.set_xlabel("Time (s)"); ax.set_yticks([]); ax.set_title(f"dF/F ({n} cells)")
plt.show()

In [ ]:
# Save the effective ops (for reproducibility / reuse as OPS_FILE)
def _ser(v):
    return v.tolist() if isinstance(v, np.ndarray) else (str(v) if isinstance(v, Path) else v)

out = Path(db["save_path0"]) / "suite2p_mesoscope_ops_used.json"
with open(out, "w") as f:
    json.dump({k: _ser(v) for k, v in ops.items()}, f, indent=2, default=str)
print("Saved:", out)